# 第7回 CNN：予想→実行→比較→説明

このNotebookは単独で動きます。同梱Pythonをimportしません。既存venvのカーネルを選び、再起動後に全セル実行してください。

[プログラム解説](https://nshrhm.github.io/intro-ai/chap06-part01/programs/guide.html) ／ [操作教材](https://nshrhm.github.io/intro-ai/chap06-part01/interactive.html)

人工4×4画像だけを使います。目的は畳み込みの計算と小さな学習を観察することです。GPU・外部画像・PyTorchは不要です。

## 1. 計算の定義（変更しなくてよいセル）
フィルターを反転せず積和を求めます。数値微分幅1e-6・全件一括更新は28パラメーターだけの教育用計算です。一般の大規模CNNより遅い方法です。

In [ ]:
"""第7回: 自作4×4画像でCNNの計算と小さな学習を観察する。

python cnn_basics.py --demo (--json は検証用)
入力は人工画像だけ。外部データ・GPU・追加ライブラリは不要。
学習率0.1・100更新と、未学習の端の画像の予測を比較する。
数値微分は28パラメーター限定の教育用。大規模学習向けではない。
"""
import argparse
import json
import math
import random
from copy import deepcopy

# Python Random(42): フィルター8値、バイアス2値、出力重み16値、バイアス2値。
_rng = random.Random(42)
INITIAL_WEIGHTS = tuple([_rng.uniform(-.5, .5) for _ in range(8)] + [.1, .1]
                        + [_rng.uniform(-.5, .5) for _ in range(16)] + [0., 0.])
IMAGE = [[0, 1, 0, 0] for _ in range(4)]
FILTER = [[1, -1], [1, -1]]


def matrix(a):
    if not a or not a[0] or any(len(row) != len(a[0]) for row in a):
        raise ValueError('空でない長方形の配列にしてください')
    if any(not math.isfinite(v) for row in a for v in row):
        raise ValueError('値は有限の数にしてください')
    return len(a), len(a[0])


def convolve(image, kernel, stride=1, padding=0):
    """フィルターを反転しない積和。padding は周囲に加えるゼロの幅。"""
    h, w = matrix(image)
    kh, kw = matrix(kernel)
    if not isinstance(stride, int) or stride < 1 or not isinstance(padding, int) or padding < 0:
        raise ValueError('strideは正の整数、paddingは非負整数')
    if h + 2 * padding < kh or w + 2 * padding < kw:
        raise ValueError('フィルターが入力より大きすぎます')
    def pixel(i, j):
        return image[i][j] if 0 <= i < h and 0 <= j < w else 0
    return [[sum(kernel[u][v] * pixel(i + u - padding, j + v - padding)
                 for u in range(kh) for v in range(kw))
             for j in range(0, w + 2 * padding - kw + 1, stride)]
            for i in range(0, h + 2 * padding - kh + 1, stride)]


def relu(image):
    matrix(image)
    return [[max(0., value) for value in row] for row in image]


def pool(image, mode='max', size=2, stride=1):
    h, w = matrix(image)
    if mode not in ('max', 'mean') or size < 1 or stride < 1 or min(h, w) < size:
        raise ValueError('プーリング条件を確認してください')
    def value(i, j):
        values = [image[i + u][j + v] for u in range(size) for v in range(size)]
        return max(values) if mode == 'max' else sum(values) / len(values)
    return [[value(i, j) for j in range(0, w - size + 1, stride)]
            for i in range(0, h - size + 1, stride)]


def samples(positions):
    """0=縦線、1=横線。位置は0始まり。並びは縦→横、それぞれ位置順。"""
    if any(p not in range(4) for p in positions):
        raise ValueError('位置は0〜3')
    return [([[float((j if label == 0 else i) == pos) for j in range(4)]
              for i in range(4)], label) for label in range(2) for pos in positions]


def forward(weights, image):
    if len(weights) != 28 or matrix(image) != (4, 4):
        raise ValueError('学習モデルは28パラメーター・4×4入力です')
    maps, activated, pooled, features = [], [], [], []
    for k in range(2):
        kernel = [weights[k * 4:k * 4 + 2], weights[k * 4 + 2:k * 4 + 4]]
        raw = [[v + weights[8 + k] for v in row] for row in convolve(image, kernel)]
        act = relu(raw)
        small = pool(act)
        maps.append(raw); activated.append(act); pooled.append(small)
        features.extend(v for row in small for v in row)
    logits = [sum(weights[10 + c * 8 + t] * features[t] for t in range(8))
              + weights[26 + c] for c in range(2)]
    maximum = max(logits)
    exps = [math.exp(v - maximum) for v in logits]
    logsum = maximum + math.log(sum(exps))
    return {'maps': maps, 'relu': activated, 'pooled': pooled, 'features': features,
            'logits': logits, 'probabilities': [v / sum(exps) for v in exps],
            'log_probabilities': [v - logsum for v in logits]}


def loss(weights, data):
    return sum(-forward(weights, image)['log_probabilities'][label] for image, label in data) / len(data)


def predict(weights, data):
    return [max(range(2), key=forward(weights, image)['probabilities'].__getitem__) for image, _ in data]


def train(rate=.1, steps=100, initial=INITIAL_WEIGHTS):
    """同じ初期値のコピーから全件一括更新。中央差分幅は1e-6。"""
    if not math.isfinite(rate) or rate < 0 or not isinstance(steps, int) or not 0 <= steps <= 1000:
        raise ValueError('rateは非負の有限値、stepsは0〜1000の整数')
    weights = list(initial)
    data = samples([1, 2])
    history = [loss(weights, data)]
    for _ in range(steps):
        gradients = []
        for j in range(28):
            plus, minus = weights.copy(), weights.copy()
            plus[j] += 1e-6; minus[j] -= 1e-6
            gradients.append((loss(plus, data) - loss(minus, data)) / 2e-6)
        weights = [v - rate * d for v, d in zip(weights, gradients)]
        history.append(loss(weights, data))
    return {'weights': weights, 'history': history,
            'train_predictions': predict(weights, data),
            'heldout_predictions': predict(weights, samples([0, 3]))}


def rounded(value):
    if isinstance(value, float):
        return round(value, 6)
    if isinstance(value, (tuple, list)):
        return [rounded(v) for v in value]
    if isinstance(value, dict):
        return {k: rounded(v) for k, v in value.items()}
    return value


def demo():
    raw = convolve(IMAGE, FILTER)
    fit = train()
    return rounded({'convolution': raw, 'relu': relu(raw),
                    'stride2': convolve(IMAGE, FILTER, stride=2),
                    'padding_shape': [len(convolve(IMAGE, FILTER, padding=1))] * 2,
                    'max_pool': pool(relu(raw)), 'mean_pool': pool(relu(raw), 'mean'),
                    'parameter_count': len(INITIAL_WEIGHTS),
                    'loss_before': fit['history'][0], 'loss_after': fit['history'][-1],
                    'train_before': predict(INITIAL_WEIGHTS, samples([1, 2])),
                    'train_after': fit['train_predictions'], 'heldout_after': fit['heldout_predictions'],
                    'labels': [0, 0, 1, 1]})



## 2. 基準結果を保存する
畳み込み1行目[-2,2,0]、損失約0.765126→0.130153、学習用[0,0,1,1]・確認用[1,1,1,1]を確認します。0=縦、1=横。基準結果は後の実験で上書きしません。

In [ ]:
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))

## 3. 学習前後の重みと画像ごとの予測
中央2位置を学習に使い、端2位置は重み更新に使いません。フィルターと損失、予測を分けて観察します。

In [ ]:
baseline_fit = train()
print('初期フィルター:', list(INITIAL_WEIGHTS[:8]))
print('学習後フィルター:', rounded(baseline_fit['weights'][:8]))
for name, data in [('学習用', samples([1,2])), ('確認用', samples([0,3]))]:
    for (image, label), before, after in zip(data, predict(INITIAL_WEIGHTS, data), predict(baseline_fit['weights'], data)):
        print(name, '画像=', image, '正解=', label, '学習前=', before, '学習後=', after)

## 実験A：フィルターの符号を反転
予想：どの応答が正・負になるでしょう？ `sign` だけを1→-1へ変えます。元データのコピーを使います。期待：1行目[-2,2,0]→[2,-2,0]。ReLUを通したら何が残るか説明してください。

In [ ]:
sign = -1
trial_image = deepcopy(IMAGE)
trial_filter = [[sign*v for v in row] for row in deepcopy(FILTER)]
print('元:', convolve(IMAGE, FILTER))
print('変更:', convolve(trial_image, trial_filter))

## 実験B：移動幅だけ変える
予想：出力は何行何列？ `trial_stride` を1と2で比較します。期待：3×3→2×2、変更後は[[-2,0],[-2,0]]。計算しなくなった位置を説明してください。

In [ ]:
trial_stride = 2
print('基準:', convolve(IMAGE, FILTER))
print('変更:', convolve(deepcopy(IMAGE), deepcopy(FILTER), stride=trial_stride))

## 実験C：最大から平均へ
予想：領域[[0,2],[0,2]]の結果は？ `trial_mode` をmaxとmeanで比較します。期待：2→1。残す情報の違いを説明してください。

In [ ]:
trial_mode = 'mean'
trial_features = relu(convolve(deepcopy(IMAGE), deepcopy(FILTER)))
print('最大:', pool(trial_features, 'max'))
print('変更:', pool(trial_features, trial_mode))

## 実験D：学習率0と0.1
予想：学習率0では重みが変わりますか？同じ初期値のコピーから100回更新します。0では初期損失と同じ、0.1では約0.130153。学習用での成功と確認用の誤りを両方説明してください。

In [ ]:
for trial_rate in [0, .1]:
    trial_fit = train(rate=trial_rate, initial=list(INITIAL_WEIGHTS))
    print('学習率:', trial_rate, '損失:', round(trial_fit['history'][-1],6), '学習用:', trial_fit['train_predictions'], '確認用:', trial_fit['heldout_predictions'])

## 振り返りと復帰
変えた条件、予想、観察結果、理由を説明しましょう。分からなくなったらカーネル再起動→全セル実行。コードを壊した場合は元のNotebookを再ダウンロードします。人工8画像での結果を、実写真の認識性能に一般化しないでください。